# Notebook 03a — Baselines & Model Zoo

**Question:** How do models perform out of the box on all 3 problem framings?

No SMOTE, no undersampling, no tuning — just `class_weight='balanced'` and default hyperparameters.
This establishes the reference point for everything that follows.

**Inputs:** `artifacts/` from nb02
**Outputs:** `results/ml_baselines.csv`


In [ ]:
import warnings; warnings.filterwarnings("ignore")
import numpy as np, pandas as pd, time, os, joblib
import matplotlib.pyplot as plt, seaborn as sns
from collections import OrderedDict
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, ExtraTreesClassifier
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.dummy import DummyClassifier
import xgboost as xgb
import lightgbm as lgb
from catboost import CatBoostClassifier

plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['figure.dpi'] = 100
sns.set_style("whitegrid")

SEED = 42
np.random.seed(SEED)
os.makedirs('results', exist_ok=True)

print("Imports OK")


## 0. Load Artifacts

In [ ]:
# Load all preprocessed splits from nb02
X_train   = joblib.load('../artifacts/X_train.pkl').astype(np.float32)
X_val     = joblib.load('../artifacts/X_val.pkl').astype(np.float32)
X_test    = joblib.load('../artifacts/X_test.pkl').astype(np.float32)
y_train   = joblib.load('../artifacts/y_train.pkl')
y_val     = joblib.load('../artifacts/y_val.pkl')
y_test    = joblib.load('../artifacts/y_test.pkl')

X_train_3c = joblib.load('../artifacts/X_train_3c.pkl').astype(np.float32)
X_val_3c   = joblib.load('../artifacts/X_val_3c.pkl').astype(np.float32)
y_train_3c = joblib.load('../artifacts/y_train_3c.pkl')
y_val_3c   = joblib.load('../artifacts/y_val_3c.pkl')

X_train_bin = joblib.load('../artifacts/X_train_bin.pkl').astype(np.float32)
X_val_bin   = joblib.load('../artifacts/X_val_bin.pkl').astype(np.float32)
y_train_bin = joblib.load('../artifacts/y_train_bin.pkl')
y_val_bin   = joblib.load('../artifacts/y_val_bin.pkl')

le_target = joblib.load('../artifacts/label_encoder_target.pkl')
le_3class = joblib.load('../artifacts/label_encoder_3class.pkl')
feature_names = joblib.load('../artifacts/feature_names.pkl')

# Merge train+val for CV (CV creates its own internal folds)
X_4c = np.vstack([X_train, X_val]);     y_4c = np.concatenate([y_train, y_val])
X_3c = np.vstack([X_train_3c, X_val_3c]); y_3c = np.concatenate([y_train_3c, y_val_3c])
X_bin = np.vstack([X_train_bin, X_val_bin]); y_bin = np.concatenate([y_train_bin, y_val_bin])

print(f'4-class: {X_4c.shape} — {le_target.classes_.tolist()}')
print(f'3-class: {X_3c.shape} — {le_3class.classes_.tolist()}')
print(f'Binary:  {X_bin.shape} — [failed, success]')
print(f'Features: {len(feature_names)}')


In [ ]:
# Experiment tracker
SCOREBOARD = []

def log_exp(phase, model, strategy, cv_mean, cv_std, n_train, notes=""):
    SCOREBOARD.append(dict(
        phase=phase, model=model, strategy=strategy,
        n_features=len(feature_names),
        cv_f1_mean=round(cv_mean, 4), cv_f1_std=round(cv_std, 4),
        n_train=n_train, notes=notes
    ))
    print(f"  {model:20s} | {strategy:10s} | F1={cv_mean:.4f} ± {cv_std:.4f}")

def run_cv(model, X, y, n_folds=5):
    cv = StratifiedKFold(n_splits=n_folds, shuffle=True, random_state=SEED)
    scores = cross_val_score(model, X, y, cv=cv, scoring="f1_macro", n_jobs=-1)
    return scores.mean(), scores.std()


## 1. Dummy Baselines

These establish the absolute floor. A model that always predicts "operating" (majority class) gets ~80% accuracy but terrible macro F1 because it ignores 3 out of 4 classes.


In [ ]:
print("=== Dummy Baselines (4-class) ===")
for name, mdl in [
    ("DummyMajority", DummyClassifier(strategy="most_frequent")),
    ("DummyStratified", DummyClassifier(strategy="stratified", random_state=SEED)),
]:
    m, s = run_cv(mdl, X_4c, y_4c)
    log_exp("baseline", name, "4class", m, s, len(X_4c))


## 2. Model Zoo — 4-class

All models with default hyperparameters and `class_weight='balanced'` (where available).
This is the raw 4-class problem with 80% operating — we expect modest results.


In [ ]:
print("=== Model Zoo — 4-class (66k rows, 80% operating) ===")

zoo_4class = OrderedDict([
    ("LogReg", LogisticRegression(class_weight="balanced", max_iter=1000, random_state=SEED, n_jobs=-1)),
    ("KNN-5", KNeighborsClassifier(n_neighbors=5, n_jobs=-1)),
    ("RF", RandomForestClassifier(n_estimators=200, class_weight="balanced", random_state=SEED, n_jobs=-1)),
    ("ExtraTrees", ExtraTreesClassifier(n_estimators=200, class_weight="balanced", random_state=SEED, n_jobs=-1)),
    ("XGBoost", xgb.XGBClassifier(n_estimators=200, learning_rate=0.1, max_depth=6,
        random_state=SEED, n_jobs=-1, verbosity=0, eval_metric="mlogloss")),
    ("LightGBM", lgb.LGBMClassifier(n_estimators=200, class_weight="balanced",
        random_state=SEED, n_jobs=-1, verbose=-1)),
    ("CatBoost", CatBoostClassifier(iterations=200, auto_class_weights="Balanced",
        random_seed=SEED, verbose=0)),
])

for name, mdl in zoo_4class.items():
    t0 = time.time()
    m, s = run_cv(mdl, X_4c, y_4c)
    log_exp("baseline", name, "4class", m, s, len(X_4c), f"{time.time()-t0:.0f}s")


## 3. Model Zoo — 3-class (no operating)

Same models, but on the 3-class dataset (closed / acquired / ipo).
By removing the noisy "operating" class, we expect a significant jump in F1.


In [ ]:
print("=== Model Zoo — 3-class (13k rows, closed/acquired/ipo) ===")

zoo_3class = OrderedDict([
    ("LogReg", LogisticRegression(class_weight="balanced", max_iter=1000, random_state=SEED, n_jobs=-1)),
    ("RF", RandomForestClassifier(n_estimators=200, class_weight="balanced", random_state=SEED, n_jobs=-1)),
    ("ExtraTrees", ExtraTreesClassifier(n_estimators=200, class_weight="balanced", random_state=SEED, n_jobs=-1)),
    ("XGBoost", xgb.XGBClassifier(n_estimators=200, learning_rate=0.1, max_depth=6,
        random_state=SEED, n_jobs=-1, verbosity=0, eval_metric="mlogloss")),
    ("LightGBM", lgb.LGBMClassifier(n_estimators=200, class_weight="balanced",
        random_state=SEED, n_jobs=-1, verbose=-1)),
    ("CatBoost", CatBoostClassifier(iterations=200, auto_class_weights="Balanced",
        random_seed=SEED, verbose=0)),
    ("SVM", SVC(C=1.0, kernel="rbf", class_weight="balanced", random_state=SEED)),
])

for name, mdl in zoo_3class.items():
    t0 = time.time()
    m, s = run_cv(mdl, X_3c, y_3c)
    log_exp("baseline", name, "3class", m, s, len(X_3c), f"{time.time()-t0:.0f}s")


## 4. Model Zoo — Binary (success vs failed)

Same models on the binary formulation: success (acquired + IPO) vs failed (closed).
This is the most practically useful framing and the easiest to model.


In [ ]:
print("=== Model Zoo — Binary (13k rows, success vs failed) ===")

zoo_binary = OrderedDict([
    ("LogReg", LogisticRegression(class_weight="balanced", max_iter=1000, random_state=SEED, n_jobs=-1)),
    ("RF", RandomForestClassifier(n_estimators=200, class_weight="balanced", random_state=SEED, n_jobs=-1)),
    ("ExtraTrees", ExtraTreesClassifier(n_estimators=200, class_weight="balanced", random_state=SEED, n_jobs=-1)),
    ("XGBoost", xgb.XGBClassifier(n_estimators=200, learning_rate=0.1, max_depth=6,
        random_state=SEED, n_jobs=-1, verbosity=0, eval_metric="logloss")),
    ("LightGBM", lgb.LGBMClassifier(n_estimators=200, class_weight="balanced",
        random_state=SEED, n_jobs=-1, verbose=-1)),
    ("CatBoost", CatBoostClassifier(iterations=200, auto_class_weights="Balanced",
        random_seed=SEED, verbose=0)),
    ("SVM", SVC(C=1.0, kernel="rbf", class_weight="balanced", random_state=SEED)),
])

for name, mdl in zoo_binary.items():
    t0 = time.time()
    m, s = run_cv(mdl, X_bin, y_bin)
    log_exp("baseline", name, "binary", m, s, len(X_bin), f"{time.time()-t0:.0f}s")


## 5. Results Summary

In [ ]:
sb = pd.DataFrame(SCOREBOARD)

# Heatmap: model × strategy
pivot = sb[sb['model'].isin(['LogReg','RF','ExtraTrees','XGBoost','LightGBM','CatBoost','SVM'])]
pivot = pivot.pivot_table(values='cv_f1_mean', index='model', columns='strategy', aggfunc='first')
pivot = pivot.reindex(columns=['4class', '3class', 'binary'])

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Heatmap
sns.heatmap(pivot, annot=True, fmt='.4f', cmap='YlGn', ax=axes[0],
            linewidths=0.5, cbar_kws={'label': 'Macro F1'})
axes[0].set_title('Model × Strategy: Macro F1 Scores')
axes[0].set_ylabel(''); axes[0].set_xlabel('')

# Bar chart: best model per strategy
best_per_strat = sb.groupby('strategy').apply(lambda x: x.loc[x['cv_f1_mean'].idxmax()])
best_per_strat = best_per_strat.sort_values('cv_f1_mean')
colors = {'4class': '#888780', '3class': '#534AB7', 'binary': '#1D9E75'}
c = [colors.get(s, '#888780') for s in best_per_strat['strategy']]
bars = axes[1].barh(
    best_per_strat['strategy'] + '\n(' + best_per_strat['model'] + ')',
    best_per_strat['cv_f1_mean'], color=c, edgecolor='white', height=0.5)
for bar, val in zip(bars, best_per_strat['cv_f1_mean']):
    axes[1].text(val + 0.005, bar.get_y() + bar.get_height()/2,
                 f'{val:.4f}', va='center', fontsize=11)
axes[1].set_xlabel('Macro F1')
axes[1].set_title('Best Model per Strategy')
axes[1].set_xlim(0, 0.85)

plt.tight_layout()
plt.savefig('results/nb03a_baselines_summary.png', bbox_inches='tight')
plt.show()

print('\nKey finding: Problem framing matters more than model selection.')
print(f'  4-class best: {pivot["4class"].max():.4f}')
print(f'  3-class best: {pivot["3class"].max():.4f}')
print(f'  Binary best:  {pivot["binary"].max():.4f}')


In [ ]:
# Save scoreboard
sb.to_csv('results/ml_baselines.csv', index=False)
print(f'Saved: results/ml_baselines.csv ({len(sb)} experiments)')
print()
print(sb.sort_values('cv_f1_mean', ascending=False).to_string(index=False))
